# Demo: Evaluación del Sistema RAG

Este notebook demuestra cómo evaluar el sistema GraphRAG.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.agents import AgenticRAG
from graphrag.evaluation.evaluator import RAGEvaluator

## 1. Inicializar sistema

In [ ]:
neo4j = Neo4jManager()
agentic_rag = AgenticRAG(neo4j)
evaluator = RAGEvaluator(agentic_rag)

## 2. Definir casos de prueba

In [ ]:
test_cases = [
    {
        "question": "Where was Einstein born?",
        "ground_truth": "Albert Einstein was born in Ulm, Germany in 1879."
    },
    {
        "question": "What was Einstein's main contribution to physics?",
        "ground_truth": "Einstein developed the theory of relativity, including both special relativity (1905) and general relativity (1915). He also made important contributions to quantum mechanics."
    },
    {
        "question": "Where did Einstein work before becoming famous?",
        "ground_truth": "Einstein worked at the Swiss Patent Office in Bern from 1902 to 1909."
    },
    {
        "question": "What prize did Einstein receive and why?",
        "ground_truth": "Einstein received the Nobel Prize in Physics in 1921 for his explanation of the photoelectric effect."
    },
    {
        "question": "How many locations are mentioned in connection with Einstein?",
        "ground_truth": "At least four locations are mentioned: Ulm (Germany), Bern (Switzerland), Princeton (New Jersey), and Germany in general."
    }
]

## 3. Ejecutar evaluación

In [ ]:
results_df = evaluator.evaluate_dataset(test_cases)

## 4. Analizar resultados

In [ ]:
# Mostrar tabla completa
pd.set_option('display.max_colwidth', 50)
results_df

In [ ]:
# Estadísticas por métrica
print("Estadísticas por métrica:")
print("=" * 50)
metrics = ['context_recall', 'faithfulness', 'answer_f1', 'precision', 'recall']
for metric in metrics:
    print(f"\n{metric}:")
    print(f"  Media: {results_df[metric].mean():.3f}")
    print(f"  Mediana: {results_df[metric].median():.3f}")
    print(f"  Min: {results_df[metric].min():.3f}")
    print(f"  Max: {results_df[metric].max():.3f}")

In [ ]:
# Análisis por herramienta usada
print("\nRendimiento por herramienta:")
print("=" * 50)
tool_stats = results_df.groupby('tool_used')[['context_recall', 'faithfulness', 'answer_f1']].mean()
print(tool_stats)

In [ ]:
# Visualizar distribución de métricas
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Distribución de Métricas de Evaluación', fontsize=16)

# Context Recall
axes[0, 0].hist(results_df['context_recall'], bins=10, edgecolor='black')
axes[0, 0].set_title('Context Recall')
axes[0, 0].set_xlabel('Score')
axes[0, 0].set_ylabel('Frecuencia')

# Faithfulness
axes[0, 1].hist(results_df['faithfulness'], bins=10, edgecolor='black', color='green')
axes[0, 1].set_title('Faithfulness')
axes[0, 1].set_xlabel('Score')
axes[0, 1].set_ylabel('Frecuencia')

# Answer F1
axes[1, 0].hist(results_df['answer_f1'], bins=10, edgecolor='black', color='orange')
axes[1, 0].set_title('Answer F1 Score')
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_ylabel('Frecuencia')

# Latency
axes[1, 1].hist(results_df['latency'], bins=10, edgecolor='black', color='red')
axes[1, 1].set_title('Latencia')
axes[1, 1].set_xlabel('Segundos')
axes[1, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 5. Análisis de casos individuales

In [ ]:
# Encontrar el mejor y peor caso
best_idx = results_df['answer_f1'].idxmax()
worst_idx = results_df['answer_f1'].idxmin()

print("MEJOR CASO:")
print("=" * 80)
print(f"Pregunta: {results_df.loc[best_idx, 'question']}")
print(f"\nRespuesta: {results_df.loc[best_idx, 'answer']}")
print(f"\nGround Truth: {results_df.loc[best_idx, 'ground_truth']}")
print(f"\nF1 Score: {results_df.loc[best_idx, 'answer_f1']:.3f}")
print(f"Faithfulness: {results_df.loc[best_idx, 'faithfulness']:.3f}")
print(f"Context Recall: {results_df.loc[best_idx, 'context_recall']:.3f}")

print("\n" + "=" * 80)
print("PEOR CASO:")
print("=" * 80)
print(f"Pregunta: {results_df.loc[worst_idx, 'question']}")
print(f"\nRespuesta: {results_df.loc[worst_idx, 'answer']}")
print(f"\nGround Truth: {results_df.loc[worst_idx, 'ground_truth']}")
print(f"\nF1 Score: {results_df.loc[worst_idx, 'answer_f1']:.3f}")
print(f"Faithfulness: {results_df.loc[worst_idx, 'faithfulness']:.3f}")
print(f"Context Recall: {results_df.loc[worst_idx, 'context_recall']:.3f}")

## 6. Guardar resultados

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"evaluation_results_{timestamp}.csv"
results_df.to_csv(filename, index=False)
print(f"Resultados guardados en: {filename}")

In [ ]:
neo4j.close()